In [1]:
import requests
import json
import time
import os
import pandas as pd

In [5]:
names = os.listdir(r"D:\PycharmProjects\deeplearning_25spr\data\touhou_split\train")
with open("tags.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(names))

In [ ]:

def read_tags_from_file(filename="tags.txt"):
    """从文件中读取标签，每行一个标签。"""
    try:
        with open(filename, 'r') as f:
            tags = [line.strip() for line in f if line.strip()]
        return tags
    except FileNotFoundError:
        print(f"错误：文件 '{filename}' 未找到。")
        return

def query_tag_image_count(tag):
    """调用Danbooru API查询标签对应的图片数量。"""
    base_url = "https://danbooru.donmai.us"
    endpoint = "/tags.json"
    params = {"search[name_matches]": tag}

    try:
        response = requests.get(base_url + endpoint, params=params)
        response.raise_for_status()  # 检查请求是否成功

        data = response.json()
        if data:
            # API返回一个包含匹配标签信息的列表，我们取第一个匹配项
            return data[0].get('post_count', 0)
        else:
            print(f"未找到标签 '{tag}'。")
            return 0
    except requests.exceptions.RequestException as e:
        print(f"查询标签 '{tag}' 时发生错误: {e}")
        return None
    except json.JSONDecodeError:
        print(f"解析标签 '{tag}' 的响应时发生错误。")
        return None
    
def query_combined_tag_image_count(combined_tags):
    """调用Danbooru API查询组合标签对应的图片数量。"""
    base_url = "https://danbooru.donmai.us"
    endpoint = "/posts.json"
    params = {"tags": combined_tags, "limit": 0}  # 设置 limit 为 0 只获取计数

    try:
        response = requests.get(base_url + endpoint, params=params)
        response.raise_for_status()  # 检查请求是否成功

        # 图片总数在响应头的 'X-Total-Count' 中
        total_count = response.headers.get('X-Total-Count')
        if total_count is not None:
            return int(total_count)
        else:
            print(f"无法获取组合标签 '{combined_tags}' 的图片数量。")
            return 0
    except requests.exceptions.RequestException as e:
        print(f"查询组合标签 '{combined_tags}' 时发生错误: {e}")
        return None



In [ ]:
tag_filename = "tags.txt"  # 替换为您的标签文件名
tags = read_tags_from_file(tag_filename)

if not tags:
    print("没有读取到任何标签，脚本终止。")
    exit()

total_image_count = 0
tag_counts = {}

print("开始查询标签...")
for tag in tags:
    print(f"正在查询标签: {tag}")
    image_count = query_tag_image_count(tag)

    if image_count is not None:
        tag_counts[tag] = image_count
        total_image_count += image_count
        print(f"标签 '{tag}' 对应的图片数量: {image_count}")
    time.sleep(2.5)  # 建议添加延迟以避免速率限制

print("\n--- 查询完成 ---")
print("\n各标签对应的图片数量:")
for tag, count in tag_counts.items():
    print(f"- {tag}: {count}")

print(f"\n全体标签对应的总图片数量: {total_image_count}")

开始查询标签...
正在查询标签: aki_minoriko


In [ ]:
df = pd.DataFrame(list(tag_counts.items()), columns=['tag', 'image_count'])
df.to_csv("tag_counts.csv", index=False)

# 获取统计信息
print(df.info)